# All Things AI 2026 Tutorial

This tutorial will introduce the basics of Generative Computing through a series of labs. Everything you need is in this Notebook.

During this tutorial, we will:
1. Get up an running with Mellea.
2. See the Instruct - Validate - Repair pattern in action.
3. See how Mellea can Make Small Models Great and take you from Demo to Deployment.

## Getting Started

Run the first cell during our introduction. The first cell will:
 * download an install ollama on your Colab instance
 * download the `granite` and `gpt-oss` model weights


In [ ]:
# Install ollama.
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!nohup ollama serve >/dev/null 2>&1 &

# Download the granite:3.3:8b weights.
!ollama pull granite4:latest
!ollama pull gpt-oss:20b

# install Mellea.
!uv pip install mellea[all]

# install matplotlib for plot generation.
!uv pip install matplotlib

# Get supporting files.
!wget https://nfulton.org/atai26.tar.gz
!tar xvfz atai26.tar.gz

# Some UI niceness.
from IPython.display import HTML, display  # noqa: E402
def set_css():
    display(HTML("\n<style>\n pre{\n white-space: pre-wrap;\n}\n</style>\n"))
get_ipython().events.register("pre_run_cell", set_css)

# Lab 1: Hello, Mellea!

Running `mellea.start_session()` initialize a new `MelleaSession`. The session holds three things:
1. The model to use for this session. In this tutorial we will use granite3.3:8b.
2. An inference engine; i.e., the code that actually calls our model. We will be using ollama, but you can also use Huggingface or any OpenAI-compatible endpoint.
3. A `Context`, which tells Mellea how to remember context between requests. This is sometimes called the "Message History" in other frameworks. Throughout this tutorial, we will be using a `SimpleContext`. In `SimpleContext`s, **every request starts with a fresh context**. There is no preserved chat history between requests. Mellea provides other types of context, but today we will not be using those features. See the Tutorial for further details.

In [ ]:
import mellea

m = mellea.start_session()

answer = m.chat(
    "tell me some fun trivia about IBM and the early history of AI."
)
print(answer.content)

# Lab 2: Instruct-Validate-Repair

Instruct-Validate-Repair is a design pattern for building robust automation using LLMs. The idea is simple:
1. Instruct the model to perform a task and specify requirements on the output of the task.
2. Validate that these requirements are satisfied by the model's output.
3. If any requirements fail, try to repair.

In [ ]:
import mellea
from mellea.stdlib.requirements import check, req, simple_validate
from mellea.stdlib.sampling import RejectionSamplingStrategy

requirements = [
    req("The email should have a salutation"),  # == r1
    req(
        "Use only lower-case letters",
        validation_fn=simple_validate(lambda x: x.lower() == x),
    ),  # == r2
    check("Do not mention purple elephants."),  # == r3
]


def write_email(m: mellea.MelleaSession, name: str, notes: str) -> str:
    email_candidate = m.instruct(
        "Write an email to {{name}} using the notes following: {{notes}}.",
        requirements=requirements,
        strategy=RejectionSamplingStrategy(loop_budget=5),
        user_variables={"name": name, "notes": notes},
        return_sampling_results=True,
    )
    if email_candidate.success:
        return str(email_candidate.result)
    else:
        return email_candidate.sample_generations[0].value


m = mellea.start_session()
print(
    write_email(
        m,
        "Olivia",
        "Olivia helped the lab over the last few weeks by organizing intern events, advertising the speaker series, and handling issues with snack delivery.",
    )
)

**Making Small Models Rock: Construction BOM Estimate with Chart Construction**

Mellea lets you do Big Model Things with Small Models. With a bit of effort, you can do with a 1B paramter on your laptop what would otherwise require GPT 5.2 with Extended Thinking.

The Open Weight Ecosystem produces really amazing small models. Thereofre, using Small MOdels (plus Mellea) to do Bith Model Things has three major advantages:
1. Low and Predictable Margins
2. Data Sovreignty
3. Local or Vendor Agnostic Inference

Doing Big Models Things with Small Models requires taking a few steps that Mellea is engineered to support:

 * **Decompose the Problem**. Example: even GPT 4.1 would beneift significantly from separating data extraction from chart construction. Breaking 
 * **Externalize Control Flow**. Small models can do a few things at time, but struggle with following lots of instructions in a single large prompt. The good news is that programming has never been more automated. Using claude inter alia to 
 * **Modularize Model Capabilities**. It's really hard to make a _general purpose_ small model that is as good as a large foundation model. However, it is much easier to make small models that are as good as large models at _specific, but still quite general, tasks_. Mixing these capabilities is hard, but that's something we can do programmatically!

In this case study, we'll build a workflow that produces a cost estimate from a bill of materials for a construction project.

Inputs:
 * PDFs containing construction plans
 * PDFs containing product catalogs

The Plan:
1. Load all documents using Docling
2. Extract the BOM from the construction plans
3. Loop over all of the items and find the relevant catalog document, extract the price, and assign category
4. Generate pie chart showing cost breakdown

### Step 1: Extracting a Bill of Mateirals from Construction Documents

We'll start by parsing the tables in the construction plan to build a bill of materials.

In [ ]:
from mellea.stdlib.components.docs.richdocument import RichDocument
construction_plans = RichDocument.from_document_file("construction_plans.pdf")
print(construction_plans.get_tables()[0].to_markdown())
print(construction_plans.get_tables()[1].to_markdown())

Now we need to extract only the tables that are part of the bill of materials. We'll do this by defining some data types and parsing our tables into those data types.

In [ ]:
from typing import Literal
from mellea.core import ModelOutputThunk
from mellea.stdlib.components.docs.richdocument import Table
from mellea.stdlib.requirements import req, simple_validate
import pydantic


class BOMEntry(pydantic.BaseModel):
    item: str
    quantity: int | str
    notes: str
    category: Literal["lumber", "windows", "doors", "other"]

class BOM(pydantic.BaseModel):
    items: list[BOMEntry]

def _bom_entry_is_well_formed(entry: BOMEntry) -> bool:
    """Checks that the BOMEntry quantity is either an integer or 'allowance'."""
    try:
        int(entry.quantity)
        return True
    except ValueError as e:
        if entry.quantity.lower() == "allowance":
            return True
    return False

def _bom_entries_are_well_formed(s: str) -> bool:
    try:
        bom = BOM.model_validate_json(s)
        return all([_bom_entry_is_well_formed(entry) for entry in bom.items])
    except pydantic.ValidationError as e:
        print(f"Failed on table: {s}")
        return False

# Filter out tables that are not lists of construction items.
@mellea.generative
def is_material_list(table_markdown: str) -> Literal["yes", "no"]:
    """Determines if the table contains a list of construction items."""

async def extract_bom(doc: RichDocument):
    bom_routines = list()
    # Fire off async requests for each table.
    for table in doc.get_tables():
        if is_material_list(m, table_markdown=table.to_markdown()) == "yes":
            next_sub_bom = m.ainstruct(
                "Reformat this table to have four columns: item, quantity, type, and notes (optional).",
                grounding_context={'table': table.to_markdown()},
                requirements=[
                    req(
                        "Quantity row should only contain an integer or Allowance",
                        validation_fn=simple_validate(_bom_entries_are_well_formed)
                    ),
                    req(
                        "type should be one of: lumber, windows, doors, other",
                        validation_fn=simple_validate(lambda x: True)
                    ), # note: this is enforced by the Literal type so no check is required.
                ],
                format=BOM
            )
            bom_routines.append(next_sub_bom)
    
    # wait for all of the async work to finish, then concatenate the results.
    bom_thunks: list[ModelOutputThunk] = [await bom_routine for bom_routine in bom_routines]
    boms = [BOM.model_validate_json(await bom_thunk.avalue()) for bom_thunk in bom_thunks]
    
    # Concatente all of the indiviual BOMs into one large list.
    all_items = []
    for bom in boms:
        all_items.extend(bom.items)
    full_bom = BOM(items=all_items)
    return full_bom

bom = None
bom = await extract_bom(doc=construction_plans)

Let's take a look at some of our extracted items:

In [ ]:
# print out some random selection of items as a markdown table.
import random
random_entries = random.choices(bom.items, k=5)
items = {f"item_{i}": entry.model_dump_json() for i, entry in enumerate(random_entries)}
print(m.instruct("Format these items as as a markdown table.", grounding_context=items).value)

We now have our entire bill of materials in a structured format. Let's move on to extracting pricing estimates!

### Step 2: Find Prices for Materials

LLMs make errors confidently. If we ask a model to generate prices for our bill of materials, the model will either hallucinate or execute a web search. The web search path may appear to be a panacea, but in practice runs into many issues -- especially in B2B settings.

A better solution is to ensure that answers are grounded in context. We can do this by leveraging the Granite RAG Intrinsics library, which are special-build model adapters for **calibrated** context relevance and answerability checking.

Let's assume that our builder has negotiated rates from lumber, windows, and doors -- they want to use those rates for these items. We will demonstrate this basic concept by adding pricing to three components of the procurement list.

To continue from here, we'll need to start a new session using the **huggingface** backend -- intrinsics are an experimental feature that are not yet supported by the ollama backend.

In [ ]:
from mellea.backends.model_ids import IBM_GRANITE_4_MICRO_3B
from mellea.backends.huggingface import LocalHFBackend
from mellea.backends.model_ids import IBM_GRANITE_4_MICRO_3B
from mellea.stdlib.context import ChatContext

m_hf = mellea.MelleaSession(
    backend=LocalHFBackend(model_id=IBM_GRANITE_4_MICRO_3B)
)

Before diving into the bom->price problem, let's start with a simple example of intrinsics. We'll use context relevance and answerability intrinsics to check the relevance of the door and window catalogs for various items in our bill of materials.

In [ ]:
from mellea.stdlib.components.docs import Document
from mellea.stdlib.components.docs.richdocument import RichDocument

# Load the windows and doors catalogs, and convert them to plain text (markdown) documents.
rd_doors = RichDocument.from_document_file("product_catalogs/door_product_catalog.pdf")
doors_doc = Document(text=rd_doors.to_markdown())

rd_windows = RichDocument.from_document_file("product_catalogs/north_ridge_windows.docx")
windows_doc = Document(text=rd_windows.to_markdown())

rd_lumber = RichDocument.from_document_file("product_catalogs/cone_mountain_lumber_catalog.xlsx")
lumber_doc = Document(text=rd_lumber.to_markdown())

Now let's do some context relevance and answerability checks:

In [ ]:
from mellea.stdlib.components.intrinsic.rag import check_context_relevance, check_answerability

door_catalog_context_relevance_score = check_context_relevance(
    question="What is the price of the 36x80 Aurora half-moon entry door?",
    document=doors_doc,
    context=ChatContext(),
    backend=m_hf.backend
)
print(f"Door catalog context relevance score for `What is the price of the 36x80 Aurora half-moon entry door?`: {door_catalog_context_relevance_score}")


door_catalog_answerability_score = check_answerability(
    question="What is the price of the 36x80 Aurora half-moon entry door?",
    documents=[doors_doc],
    context=ChatContext(),
    backend=m_hf.backend
)
print(f"Door catalog answerability score for `What is the price of the 36x80 Aurora half-moon entry door?`: {door_catalog_answerability_score}")

Let's see what happens if we leave the question the same but substitute the doors catalog for the lumber catalog:

In [ ]:
lumber_catalog_context_relevance_score = check_context_relevance(
    question="What is the price of the 36x80 Aurora half-moon entry door?",
    document=lumber_doc,
    context=ChatContext(),
    backend=m_hf.backend
)
print(f"Lumber catalog context_relevance score for `What is the price of the 36x80 Aurora half-moon entry door?`: {lumber_catalog_context_relevance_score}")


lumber_catalog_context_relevance_score = check_answerability(
    question="What is the price of of the 36x80 Aurora half-moon entry door?",
    documents=[lumber_doc],
    context=ChatContext(),
    backend=m_hf.backend
)
print(f"Lumber catalog answerability score for `What is the price of the 36x80 Aurora half-moon entry door?`: {lumber_catalog_context_relevance_score}")

To have extra confidence, or if we need to refer to the source material directly, we can pair these intrinscis with the  `find_citations` intrinsic, which identifies the specific segments of the document were used to answer the question:

In [ ]:
from mellea.stdlib.components.chat import Message
from mellea.stdlib.components.intrinsic.rag import find_citations
import json # for pretty-printing the citation results.

price_response = m_hf.instruct("What is the price of a half-moon entry door?", grounding_context={"doors": doors_doc.text})

# Because we are using a SimpleContext, we will need to construct a chat history containing just the most recent question and answer.
ctx = ChatContext()
ctx = ctx.add(Message(role="user", content="What is the price of a half-moon entry door?"))
ctx = ctx.add(Message(role="assistant", content=price_response.value))

citations = find_citations(
    response=price_response.value,
    documents=[doors_doc],
    context=ctx,
    backend=m_hf.backend
)

print(json.dumps(citations, indent=4))

We're ready to get back to our case study and see how intrinsics can be used in a generative program.

In [ ]:
class UnitPriceResponseFmt(pydantic.BaseModel):
    unit_price: float

class TotalPriceResponseFmt(pydantic.BaseModel):
    total_price: float

class BomEntryWithPrice(pydantic.BaseModel):
    item: str
    quantity: int | str
    notes: str
    category: Literal["lumber", "windows", "doors", "other"]
    unit_price: float | None
    total_price: float | None

def get_prices(m: mellea.MelleaSession, bom: BOM) -> list[BomEntryWithPrice]: 
    prices: list[BomEntryWithPrice] = list()
    # Go through every entry in the BOM and figure out if there's an entry in the documents.
    for i, entry in enumerate(bom.items):
        print(f"{i}/{len(bom.items)} ({entry.category})")
        catalog = None
        if entry.category == "windows":
            catalog = windows_doc
        elif entry.category == "doors":
            catalog = doors_doc
        elif entry.category == "lumber":
            # catalog = lumber_doc
            pass # sic: we'll skip lumber so that the runtime is reasonable on colab T4.
        else:
            pass 
        
        if catalog:
            answerability_score = check_answerability(
                f"What is the price of {entry.item}?", 
                documents=[catalog], 
                context=ChatContext(),
                backend=m.backend
            )
            if answerability_score > 0.80:
                unit_price_response = m.instruct(
                    f"Find the `unit_price` of {entry.item} in the catalog.",
                    grounding_context={"catalog": catalog.text},
                    format=UnitPriceResponseFmt,
                )
                print(unit_price_response.value)
                unit_price = UnitPriceResponseFmt.model_validate_json(unit_price_response.value).unit_price
                
                total_price_respsone = m.instruct(f"Find the `total_price` given the `unit_price` and specified `quantity` for {entry.item}",
                                         grounding_context={
                                             "unit_price": str(unit_price),
                                             "quantity": str(entry.quantity)
                                         },
                                         format=TotalPriceResponseFmt)
                total_price = TotalPriceResponseFmt.model_validate_json(total_price_respsone.value).total_price
                
                prices.append(BomEntryWithPrice(
                    item=entry.item,
                    quantity=entry.quantity,
                    notes=entry.notes,
                    category=entry.category,
                    unit_price=unit_price,
                    total_price=total_price,
                ))
                continue
        
        # If we made it here then we did not succeed in adding this item's price from the catalog.
        # Exercise: use web_search tool and citation intrinsicto find prices and validate groundedness.
        # For now, we'll add back the original item with unknown prices.
        prices.append(BomEntryWithPrice(
            item=entry.item,
            quantity=entry.quantity,
            notes=entry.notes,
            category=entry.category,
            unit_price=None,
            total_price=None,
        ))
    return prices

prices = get_prices(m, bom)

In [ ]:
# Exercise: write a requirement that enforces this constraint. Remember that quantity might be "allowance"!
print(f"Calculated price: {prices[-4].unit_price * prices[-4].quantity}")
print(f"LLM's price estimation: {prices[-4].total_price}")

We'll stop at doors and windows to keep this tutorial brief.

_take-home exercise:_ add back lumber.

_take-home exercise:_ use Mellea's search tool to add pricing estimates for other items in the BOM.

### Step 4: Building the Final Report

In [ ]:
from mellea.backends.model_ids import OPENAI_GPT_OSS_20B
from mellea.backends.model_options import ModelOption
from mellea.stdlib.tools import local_code_interpreter
from mellea.backends.tools import MelleaTool


m = mellea.start_session(backend_name="ollama", model_id=OPENAI_GPT_OSS_20B)

report_grounding_context = {
    x.item: json.dumps({
        "total_price": x.total_price if x.total_price is not None else "unknown", 
        "category": x.category
    })
    for x in prices
}


pie_chart_result = m.instruct("Use the code interpreter tool to create a pie chart of known cost breakdowns by category. Put the pie chart in /tmp/chart.png", 
           grounding_context=report_grounding_context, tool_calls=True,
           model_options={ModelOption.TOOLS: [MelleaTool.from_callable(local_code_interpreter)]})

pie_chart_result.tool_calls['local_code_interpreter'].call_func()

report = m.instruct(
    "Write an HTML report with a top-line cost breakdown by category and a line-item material list with prices. At the top include the /tmp/chart.png image.", 
    grounding_context=report_grounding_context
)

open("/tmp/report.html", "w").write(report.value)